<a href="https://colab.research.google.com/github/eshfat786-prog/sp500-eps-forecasting-backtest/blob/main/sp500-eps-forecasting-backtest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install prophet statsforecast scipy statsmodels

In [5]:
import logging
from pathlib import Path
import warnings
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
from prophet import Prophet
from scipy.stats import t
from statsforecast.models import AutoARIMA
from statsmodels.stats.multitest import multipletests

logging.disable(logging.CRITICAL)

d = pd.read_csv(
    'sp500_20_eps_panel.csv', parse_dates=['fiscalDateEnding', 'reportedDate']
)
d['Quarter'] = d.fiscalDateEnding.dt.to_period('Q')
test = d[d.Quarter >= '2020Q1'].copy()
assert len(test) == 540

x = d[
    (d.Ticker == 'PEP')
    & d.Quarter.isin(pd.PeriodIndex(['2016Q2', '2016Q4'], freq='Q'))
].reportedEPS.to_numpy()
assert len(x) == 2

row = dict(
    Ticker='PEP',
    Sector='Consumer Staples',
    fiscalDateEnding=pd.Timestamp('2016-09-30'),
    Quarter=pd.Period('2016Q3'),
    reportedEPS=x.mean(),
)
d = pd.concat([d, pd.DataFrame([row])], ignore_index=True)


def fit(g):
    g = g.sort_values('Quarter').reset_index(drop=True)
    assert list(g.Quarter) == list(pd.period_range('2012Q1', '2026Q3', freq='Q'))
    out = []
    for j in range(32, 59):
        h = g.iloc[:j]
        r = g.iloc[j]
        a = AutoARIMA(
            season_length=4, ic='aic', stepwise=True, approximation=False
        ).fit(h.reportedEPS.to_numpy())
        p = Prophet(
            yearly_seasonality=2,
            weekly_seasonality=False,
            daily_seasonality=False,
            uncertainty_samples=0,
        ).add_seasonality('quarterly', 365.25 / 4, 1, prior_scale=0.1)
        p.fit(
            h[['fiscalDateEnding', 'reportedEPS']].rename(
                columns={'fiscalDateEnding': 'ds', 'reportedEPS': 'y'}
            ),
            seed=42,
        )
        v = float(
            p.predict(pd.DataFrame({'ds': [r.fiscalDateEnding]})).yhat.iloc[0]
        )
        u = float(a.predict(1)['mean'][0])
        out.append(
            dict(
                Ticker=r.Ticker,
                Quarter=r.Quarter,
                SARIMA_EPS=u,
                Prophet_EPS=v,
                SARIMA_order=str(a.model_['arma']),
                SARIMA_AIC=a.model_['aic'],
                training_quarters=j,
                latest_training_reportedDate=h.reportedDate.max(),
            )
        )
    return out


if __name__ == '__main__':
    with warnings.catch_warnings(record=True):
        warnings.simplefilter('ignore')
        with ProcessPoolExecutor(max_workers=2) as pool:
            rows = [
                r
                for rr in pool.map(fit, [g for _, g in d.groupby('Ticker')])
                for r in rr
            ]

    b = test.merge(
        pd.DataFrame(rows), on=['Ticker', 'Quarter'], validate='one_to_one'
    )
    pd.testing.assert_frame_equal(
        b[test.columns].reset_index(drop=True), test.reset_index(drop=True)
    )
    b['Hybrid_Prophet'] = (b.estimatedEPS + b.Prophet_EPS) / 2
    b['Hybrid_SARIMA'] = (b.estimatedEPS + b.SARIMA_EPS) / 2

    names = {
        'estimatedEPS': 'Human Consensus',
        'SARIMA_EPS': 'SARIMA',
        'Prophet_EPS': 'Prophet',
        'Hybrid_Prophet': 'Human + Prophet',
        'Hybrid_SARIMA': 'Human + SARIMA',
    }
    assert np.isfinite(b[list(names) + ['reportedEPS']]).all().all()

    b['Regime'] = np.where(
        b.Quarter < '2022Q1', 'Pandemic Shock', 'Post-Pandemic'
    )
    b.to_csv('backtest_results_full.csv', index=False)

    def metrics(g, k):
        a = g.reportedEPS.to_numpy()
        f = g[k].to_numpy()
        e = abs(a - f)
        den = abs(a) + abs(f)
        return [
            e.mean(),
            np.sqrt(np.mean(e**2)),
            np.divide(200 * e, den, out=np.zeros_like(e), where=den != 0).mean(),
        ]

    scores = ['MAE', 'RMSE', 'sMAPE (%)']
    pooled = []
    for k, n in names.items():
        z = [n, *metrics(b, k), np.nan, np.nan, np.nan]
        if k != 'estimatedEPS':
            diff = (b.reportedEPS - b[k]) ** 2 - (
                b.reportedEPS - b.estimatedEPS
            ) ** 2
            q = diff.groupby(b.Quarter).mean().to_numpy()
            N = len(q)
            u = q - q.mean()
            lrv = u @ u / N + 2 * sum(
                (1 - l / 4) * (u[l:] @ u[:-l]) / N for l in range(1, 4)
            )
            stat = q.mean() / np.sqrt(lrv / N * N / (N - 1))
            z[4:6] = [stat, 2 * t.sf(abs(stat), N - 1)]
        pooled.append(z)

    p = pd.DataFrame(pooled, columns=['Model', *scores, 'DM', 'p', 'Holm p'])
    p.loc[1:, 'Holm p'] = multipletests(p.loc[1:, 'p'], method='holm')[1]
    print(p.to_string(index=False, float_format=lambda x: f'{x:.5f}'))

    for group in ['Sector', 'Regime']:
        print(
            pd.DataFrame(
                [
                    [label, len(g), n, *metrics(g, k)]
                    for label, g in b.groupby(group)
                    for k, n in names.items()
                ],
                columns=[group, 'N', 'Model', *scores],
            ).to_string(index=False, float_format=lambda x: f'{x:.3f}')
        )

    print(
        'timing',
        sum(b.latest_training_reportedDate >= b.reportedDate),
        sum(b.latest_training_reportedDate > b.fiscalDateEnding),
        b.reportedDate.max(),
    )
    Path('backtest_imputation_log.txt').write_text(
        f'One training-only placeholder: PEP 2016Q3, fiscalDateEnding 2016-09-30. reportedEPS = ({x[0]} + {x[1]}) / 2 = {x.mean():.2f}, linear interpolation on evenly spaced quarter index. Neighboring quarters: 2016Q2 and 2016Q4. No other fields imputed. Original CSV unchanged. All 540 test rows verified unchanged in original columns.\n'
    )


          Model     MAE    RMSE  sMAPE (%)      DM       p  Holm p
Human Consensus 0.18401 0.30954   11.02035     NaN     NaN     NaN
         SARIMA 0.36783 0.68595   19.21531 3.52922 0.00157 0.00315
        Prophet 0.40336 0.69269   19.85172 4.10723 0.00035 0.00106
Human + Prophet 0.25937 0.43055   14.23952 4.25281 0.00024 0.00096
 Human + SARIMA 0.23873 0.42619   13.60810 2.76230 0.01039 0.01039
                Sector   N           Model   MAE  RMSE  sMAPE (%)
Consumer Discretionary 108 Human Consensus 0.223 0.352     16.496
Consumer Discretionary 108          SARIMA 0.433 0.671     27.639
Consumer Discretionary 108         Prophet 0.483 0.649     26.988
Consumer Discretionary 108 Human + Prophet 0.308 0.443     19.138
Consumer Discretionary 108  Human + SARIMA 0.288 0.475     19.324
      Consumer Staples 108 Human Consensus 0.120 0.278     13.776
      Consumer Staples 108          SARIMA 0.103 0.154     16.079
      Consumer Staples 108         Prophet 0.121 0.155     16.620
    